# SupplySight AI
## Enterprise Procurement Decision Intelligence Platform

### Notebook 02 : Data Understanding & Business Understanding

---

### Objective

The objective of this notebook is to understand the structure, quality, and business meaning of the available datasets before data cleaning and feature engineering.

This notebook focuses on:

- Understanding procurement data
- Profiling every dataset
- Identifying business entities
- Understanding relationships between datasets
- Identifying potential analytical opportunities
- Planning the enterprise data warehouse

In [8]:
import sys

!{sys.executable} -m pip install seaborn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\chavi\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [9]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

plt.style.use("ggplot")

print("Libraries Imported Successfully")

Libraries Imported Successfully


# 1. Define Project Directory

In [10]:
# ============================================================
# Configuration
# ============================================================

PROJECT_PATH = Path(
    r"C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform"
)

PROCESSED = PROJECT_PATH / "Datasets" / "Processed Datasets"

RAW = PROJECT_PATH / "Datasets" / "Raw Datasets"

EXTERNAL = PROJECT_PATH / "Datasets" / "External Datasets"

print("Project Paths Configured Successfully")

Project Paths Configured Successfully


# 2. Load Datasets

In [11]:
# Main Procurement Dataset 
dataco = pd.read_csv( 
    RAW / "DataCoSupplyChainDataset.csv", 
    encoding="latin1"
) 

# Data Dictionary 
description = pd.read_csv( 
    RAW / "DescriptionDataCoSupplyChain.csv", 
    encoding="latin1"
) 

# Fuel Prices 
fuel = pd.read_csv( 
    EXTERNAL / "fuel_prices.csv" 
) 

# Holiday Data 
holiday = pd.read_csv( 
    EXTERNAL / "holidays_cleaned.csv" 
) 

# Exchange Rates 
usd_inr = pd.read_csv( 
    RAW / "DEXINUS.csv"
) 

usd_eur = pd.read_csv( 
    RAW / "DEXUSEU.csv"
) 

# Commodity Prices 
commodity = pd.read_excel( 
    RAW / "CMO-Historical-Data-Monthly.xlsx", 
    sheet_name="Monthly Prices", 
    header=4 
) 

# Weather 
weather = pd.read_csv( 
    EXTERNAL / "weather.csv"
) 
print("All datasets loaded successfully.")

All datasets loaded successfully.


# 3. Dataset Inventory

In [13]:
# ============================================================
# Enterprise Dataset Inventory
# ============================================================

dataset_inventory = pd.DataFrame({

    "Dataset": [
        "ERP Transactions",
        "Data Dictionary",
        "Commodity Prices",
        "Fuel Prices",
        "USD-INR Exchange",
        "USD-EUR Exchange",
        "Weather",
        "Holiday"
    ],

    "Rows": [
        dataco.shape[0],
        description.shape[0],
        commodity.shape[0],
        fuel.shape[0],
        usd_inr.shape[0],
        usd_eur.shape[0],
        weather.shape[0],
        holiday.shape[0]
    ],

    "Columns": [
        dataco.shape[1],
        description.shape[1],
        commodity.shape[1],
        fuel.shape[1],
        usd_inr.shape[1],
        usd_eur.shape[1],
        weather.shape[1],
        holiday.shape[1]
    ]

})

print("=" * 70)
print("DATASET INVENTORY")
print("=" * 70)

display(dataset_inventory)

DATASET INVENTORY


,Dataset,Rows,Columns
0,ERP Transactions,180519,53
1,Data Dictionary,52,2
2,Commodity Prices,798,72
3,Fuel Prices,2299,5
4,USD-INR Exchange,1305,2
5,USD-EUR Exchange,1305,2
6,Weather,8766,10
7,Holiday,220,5


### Observations

- **ERP Transactions** is the primary dataset with **180,519 records and 53 columns**.
- External datasets cover **commodity prices, fuel prices, exchange rates, weather, and holidays**, enabling contextual procurement analysis.
- The datasets have different **sizes and granularities**, so date and key standardization will be required before integration.
- The overall dataset structure provides a strong foundation for **procurement, cost, supplier, and operational analysis**.

### Key Takeaway

The ERP dataset will serve as the **core analytical dataset**, enriched with external economic, weather, and calendar data for deeper business insights.

In [14]:
#Rename the first column
commodity.rename(
    columns={"Unnamed: 0": "Date"},
    inplace=True
)

In [15]:
#Remove the units row
commodity = commodity.iloc[1:].reset_index(drop=True)

In [16]:
#Convert the Date column
commodity["Date"] = pd.to_datetime(
    commodity["Date"].str.replace("M", "-"),
    format="%Y-%m"
)

In [17]:
#Check the result
commodity.head()

,Date,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI","Coal, Australian","Coal, South African **","Natural gas, US","Natural gas, Europe","Liquefied natural gas, Japan",Natural gas index,Cocoa,"Coffee, Arabica","Coffee, Robusta","Tea, avg 3 auctions","Tea, Colombo","Tea, Kolkata","Tea, Mombasa",Coconut oil,Groundnuts,Fish meal,Groundnut oil **,Palm oil,Palm kernel oil,Soybeans,Soybean oil,Soybean meal,Rapeseed oil,Sunflower oil,Barley,Maize,Sorghum,"Rice, Thai 5%","Rice, Thai 25%","Rice, Thai A.1","Rice, Viet Namese 5%","Wheat, US SRW","Wheat, US HRW","Banana, Europe","Banana, US",Orange,Beef **,Chicken **,Lamb **,"Shrimps, Mexican","Sugar, EU","Sugar, US","Sugar, world","Tobacco, US import u.v.","Logs, Cameroon","Logs, Malaysian","Sawnwood, Cameroon","Sawnwood, Malaysian",Plywood,"Cotton, A Index","Rubber, TSR20 **","Rubber, RSS3",Phosphate rock,DAP,TSP,Urea,Potassium chloride **,Aluminum,"Iron ore, cfr spot",Copper,Lead,Tin,Nickel,Zinc,Gold,Platinum,Silver
0,1960-01-01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.63,0.94,0.7,1.03,0.93,1.12,1.04,390,…,…,334,233,…,94,204,91.9,…,…,20.36,45,39,104.45,…,…,…,…,59.89,…,0.14,0.12,0.71,0.3,…,1.43,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.65,…,0.82,13,...,53,42.25,28.5,511.47,11.42,715.4,206.1,2180.4,1631,260.8,35.27,83.5,0.91
1,1960-02-01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.61,0.95,0.69,1.03,0.93,1.12,1.04,379,…,…,341,229,…,91,201,86.7,…,…,20.41,44,39,103.54,…,…,…,…,60.99,…,0.14,0.11,0.71,0.3,…,1.5,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.65,…,0.83,13,...,53,42.25,28.5,511.47,11.42,728.19,203.7,2180.4,1631,244.9,35.27,83.5,0.91
2,1960-03-01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.58,0.93,0.69,1.03,0.93,1.12,1.04,361,…,…,338,225,…,92,201,84.1,…,…,20.72,45,35,103.79,…,…,…,…,61.73,…,0.14,0.13,0.77,0.3,…,1.5,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.65,…,0.86,13,...,53,42.25,28.5,511.47,11.42,684.94,210.3,2173.8,1631,248.7,35.27,83.5,0.91
3,1960-04-01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.6,0.93,0.68,1.03,0.93,1.12,1.04,338,…,…,333,225,…,93,207,86.7,…,…,20.64,45,35,100.97,…,…,…,…,60.99,…,0.14,0.14,0.84,0.3,…,1.68,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.64,…,0.86,13,...,53,42.25,28.5,511.47,11.42,723.11,213.6,2178.2,1631,254.6,35.27,83.5,0.91
4,1960-05-01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.6,0.92,0.69,1.03,0.93,1.12,1.04,321,…,…,335,225,…,93,209,81.5,…,…,20.61,48,35,102.15,…,…,…,…,57.69,…,0.15,0.14,0.76,0.3,…,1.76,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.65,…,0.93,13,...,53,42.25,28.5,511.47,11.42,684.75,213.4,2162.7,1631,253.8,35.27,83.5,0.91


### Observations

- The commodity dataset has been successfully **cleaned and standardized**.
- The first column is renamed to `Date` and converted to a proper **datetime format**.
- The units row has been removed, leaving **monthly commodity price observations**.
- The dataset contains a wide range of **energy, agricultural, metals, and raw-material price indicators**.
- The cleaned structure is now suitable for **time-series analysis and integration with procurement data**.

# 4. Data Preview

In [18]:
datasets = {
    "DataCo": dataco,
    "Description": description,
    "Fuel": fuel,
    "Holiday": holiday,
    "Exchange USD-INR": usd_inr,
    "Exchange USD-EUR": usd_eur,
    "Commodity": commodity,
    "Weather": weather
}

for name, df in datasets.items():

    print("="*80)
    print(name.upper())
    print("="*80)

    display(df.head())

DATACO


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


DESCRIPTION


,FIELDS,DESCRIPTION
0,Type,: Type of transaction made
1,Days for shipping (real),: Actual shipping days of the purchased product
2,Days for shipment (scheduled),: Days of scheduled delivery of the purchased...
3,Benefit per order,: Earnings per order placed
4,Sales per customer,: Total sales per customer made per customer


FUEL


,Date,Region,Fuel_Type,Fuel_Price,Unit
0,2015-01-05,PADD 2,No 2 Diesel,3.102,$/GAL
1,2015-01-05,PADD 5 EXCEPT CALIFORNIA,No 2 Diesel,3.074,$/GAL
2,2015-01-05,PADD 1C,No 2 Diesel,3.109,$/GAL
3,2015-01-05,PADD 3,No 2 Diesel,3.045,$/GAL
4,2015-01-05,PADD 5,No 2 Diesel,3.220,$/GAL


... (6 outputs left out)

COMMODITY


,Date,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI","Coal, Australian","Coal, South African **","Natural gas, US","Natural gas, Europe","Liquefied natural gas, Japan",Natural gas index,Cocoa,"Coffee, Arabica","Coffee, Robusta","Tea, avg 3 auctions","Tea, Colombo","Tea, Kolkata","Tea, Mombasa",Coconut oil,Groundnuts,Fish meal,Groundnut oil **,Palm oil,Palm kernel oil,Soybeans,Soybean oil,Soybean meal,Rapeseed oil,Sunflower oil,Barley,Maize,Sorghum,"Rice, Thai 5%","Rice, Thai 25%","Rice, Thai A.1","Rice, Viet Namese 5%","Wheat, US SRW","Wheat, US HRW","Banana, Europe","Banana, US",Orange,Beef **,Chicken **,Lamb **,"Shrimps, Mexican","Sugar, EU","Sugar, US","Sugar, world","Tobacco, US import u.v.","Logs, Cameroon","Logs, Malaysian","Sawnwood, Cameroon","Sawnwood, Malaysian",Plywood,"Cotton, A Index","Rubber, TSR20 **","Rubber, RSS3",Phosphate rock,DAP,TSP,Urea,Potassium chloride **,Aluminum,"Iron ore, cfr spot",Copper,Lead,Tin,Nickel,Zinc,Gold,Platinum,Silver
0,1960-01-01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.63,0.94,0.7,1.03,0.93,1.12,1.04,390,…,…,334,233,…,94,204,91.9,…,…,20.36,45,39,104.45,…,…,…,…,59.89,…,0.14,0.12,0.71,0.3,…,1.43,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.65,…,0.82,13,...,53,42.25,28.5,511.47,11.42,715.4,206.1,2180.4,1631,260.8,35.27,83.5,0.91
1,1960-02-01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.61,0.95,0.69,1.03,0.93,1.12,1.04,379,…,…,341,229,…,91,201,86.7,…,…,20.41,44,39,103.54,…,…,…,…,60.99,…,0.14,0.11,0.71,0.3,…,1.5,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.65,…,0.83,13,...,53,42.25,28.5,511.47,11.42,728.19,203.7,2180.4,1631,244.9,35.27,83.5,0.91
2,1960-03-01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.58,0.93,0.69,1.03,0.93,1.12,1.04,361,…,…,338,225,…,92,201,84.1,…,…,20.72,45,35,103.79,…,…,…,…,61.73,…,0.14,0.13,0.77,0.3,…,1.5,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.65,…,0.86,13,...,53,42.25,28.5,511.47,11.42,684.94,210.3,2173.8,1631,248.7,35.27,83.5,0.91
3,1960-04-01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.6,0.93,0.68,1.03,0.93,1.12,1.04,338,…,…,333,225,…,93,207,86.7,…,…,20.64,45,35,100.97,…,…,…,…,60.99,…,0.14,0.14,0.84,0.3,…,1.68,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.64,…,0.86,13,...,53,42.25,28.5,511.47,11.42,723.11,213.6,2178.2,1631,254.6,35.27,83.5,0.91
4,1960-05-01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.6,0.92,0.69,1.03,0.93,1.12,1.04,321,…,…,335,225,…,93,209,81.5,…,…,20.61,48,35,102.15,…,…,…,…,57.69,…,0.15,0.14,0.76,0.3,…,1.76,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.65,…,0.93,13,...,53,42.25,28.5,511.47,11.42,684.75,213.4,2162.7,1631,253.8,35.27,83.5,0.91


WEATHER


,date,city,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,snowfall_sum,rain_sum,wind_speed_10m_max,weather_code
0,2015-01-01,New York,2.2835,-4.5665,-2.281083,0.0,0.00,0.0,21.267440,1.0
1,2015-01-02,New York,5.1335,-0.8165,1.043917,0.0,0.00,0.0,19.881649,3.0
2,2015-01-03,New York,3.6835,-3.4165,-0.447750,10.2,3.08,5.8,15.175612,75.0
3,2015-01-04,New York,13.2335,4.4335,8.625167,11.1,0.00,11.1,19.969458,61.0
4,2015-01-05,New York,11.9835,-1.6165,3.350167,2.4,0.00,2.4,26.756500,55.0


### Observations

- All **8 datasets** were successfully loaded and previewed.
- **DataCo** is the primary transaction dataset containing detailed order, customer, product, sales, and shipping information.
- External datasets provide **fuel, holiday, exchange-rate, commodity, and weather** context.
- Datasets have different **date formats and granularities**, requiring standardization before integration.
- The data structure supports analysis of **cost, profitability, delivery performance, and external risk factors**.

# 5. Dataset Profiling

In [19]:
def profile_dataset(df):

    profile = pd.DataFrame({

        "Data Type":df.dtypes,

        "Missing Values":df.isnull().sum(),

        "Missing %":round(df.isnull().mean()*100,2),

        "Unique Values":df.nunique()

    })

    return profile

In [20]:
profile_dataset(dataco)

,Data Type,Missing Values,Missing %,Unique Values
Type,str,0,0.00,4
Days for shipping (real),int64,0,0.00,7
Days for shipment (scheduled),int64,0,0.00,4
Benefit per order,float64,0,0.00,21998
Sales per customer,float64,0,0.00,2927
Delivery Status,str,0,0.00,4
Late_delivery_risk,int64,0,0.00,2
Category Id,int64,0,0.00,51
Category Name,str,0,0.00,50
Customer City,str,0,0.00,563


### Observations

- The DataCo dataset is largely complete, with most fields having **0% missing values**.
- **Order Zipcode** has significant missingness (**86.24%**) and requires review before use.
- **Product Description** is completely missing (**100%**) and is not analytically useful in its current form.
- Customer and order identifiers show high uniqueness, supporting **entity-level analysis**.
- Date fields are currently stored as strings and should be converted to **datetime** during data cleaning.
- Overall, the dataset is suitable for further analysis after addressing **high-missing fields and data types**.

# 6. Dataset Information

In [21]:
for name, df in datasets.items():

    print("="*80)

    print(name)

    print("="*80)

    print(df.info())

    print()

DataCo
<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180

### Observations

- **DataCo** is the largest dataset with **180,519 records and 53 columns**.
- Most datasets are structurally complete; **USD-INR and USD-EUR** contain some missing exchange-rate values.
- Date fields are mostly stored as **string/object types** and require datetime conversion.
- The **Commodity** dataset has 72 columns, but price fields are currently stored as `object` and require numeric conversion.
- Data types and missing values identified here will guide the **data-cleaning and transformation stage**.

# 7. Missing Value Analysis

In [22]:
missing = profile_dataset(dataco)

missing = missing.sort_values(
    by="Missing %",
    ascending=False
)

missing.head(20)

,Data Type,Missing Values,Missing %,Unique Values
Product Description,float64,180519,100.00,0
Order Zipcode,float64,155679,86.24,609
Type,str,0,0.00,4
Order Profit Per Order,float64,0,0.00,21998
Order Id,int64,0,0.00,65752
Order Item Cardprod Id,int64,0,0.00,118
Order Item Discount,float64,0,0.00,1017
Order Item Discount Rate,float64,0,0.00,18
Order Item Id,int64,0,0.00,180519
Order Item Product Price,float64,0,0.00,75


### Observations

- `Product Description` has **100% missing values** and provides no usable information.
- `Order Zipcode` has **86.24% missing values**, making it unsuitable for reliable analysis in its current form.
- All remaining fields have **0% missing values**, indicating strong overall data completeness.
- These two fields should be **reviewed for removal or alternative handling** during data cleaning.

# 8. Duplicate Analysis

In [23]:
for name, df in datasets.items():

    print(name)

    print("Duplicate Rows :", df.duplicated().sum())

    print()

DataCo
Duplicate Rows : 0

Description
Duplicate Rows : 0

Fuel
Duplicate Rows : 0

Holiday
Duplicate Rows : 0

Exchange USD-INR
Duplicate Rows : 0

Exchange USD-EUR
Duplicate Rows : 0

Commodity
Duplicate Rows : 0

Weather
Duplicate Rows : 0



### Observations

- **No duplicate rows** were identified across any of the 8 datasets.
- This indicates good **row-level data integrity** at the current stage.
- No duplicate removal is required before proceeding to **data cleaning and transformation**.

# 9. Statistical Summary

In [24]:
dataco.describe().T

,count,mean,std,min,25%,50%,75%,max
Days for shipping (real),180519.0,3.497654,1.623722,0.000000,2.000000,3.000000,5.000000,6.000000
Days for shipment (scheduled),180519.0,2.931847,1.374449,0.000000,2.000000,4.000000,4.000000,4.000000
Benefit per order,180519.0,21.974989,104.433526,-4274.979980,7.000000,31.520000,64.800003,911.799988
Sales per customer,180519.0,183.107609,120.043670,7.490000,104.379997,163.990005,247.399994,1939.989990
Late_delivery_risk,180519.0,0.548291,0.497664,0.000000,0.000000,1.000000,1.000000,1.000000
Category Id,180519.0,31.851451,15.640064,2.000000,18.000000,29.000000,45.000000,76.000000
Customer Id,180519.0,6691.379495,4162.918106,1.000000,3258.500000,6457.000000,9779.000000,20757.000000
Customer Zipcode,180516.0,35921.126914,37542.461122,603.000000,725.000000,19380.000000,78207.000000,99205.000000
Department Id,180519.0,5.443460,1.629246,2.000000,4.000000,5.000000,7.000000,12.000000
Latitude,180519.0,29.719955,9.813646,-33.937553,18.265432,33.144863,39.279617,48.781933


In [25]:
dataco.describe(include="object").T

,count,unique,top,freq
Type,180519,4,DEBIT,69295
Delivery Status,180519,4,Late delivery,98977
Category Name,180519,50,Cleats,24551
Customer City,180519,563,Caguas,66770
Customer Country,180519,2,EE. UU.,111146
Customer Email,180519,1,XXXXXXXXX,180519
Customer Fname,180519,782,Mary,65150
Customer Lname,180511,1109,Smith,64104
Customer Password,180519,1,XXXXXXXXX,180519
Customer Segment,180519,3,Consumer,93504


### Observations

- **Sales** average ~203.77, while **Order Profit** averages ~21.97, indicating positive overall profitability.
- **Late delivery risk is 54.83%**, highlighting delivery performance as a key business area.
- Profitability shows high variation, including **negative profit outliers**.
- **Standard Class** is the dominant shipping mode, while **Consumer** is the largest customer segment.
- The dataset covers **50 product categories, 164 countries, and 23 regions**, providing strong scope for segmentation analysis.
- Several fields show skewed distributions and extreme values, requiring **outlier and distribution analysis** during preprocessing.

# 10. Business Understanding

## Business Entities

The procurement ecosystem consists of the following entities:

- Customer
- Order
- Product
- Supplier
- Shipment
- Market
- Department
- Category
- Inventory
- Delivery

In [26]:
pd.DataFrame(dataco.columns, columns=["Columns"])

,Columns
0,Type
1,Days for shipping (real)
2,Days for shipment (scheduled)
3,Benefit per order
4,Sales per customer
5,Delivery Status
6,Late_delivery_risk
7,Category Id
8,Category Name
9,Customer City


### Observation

- The dataset captures key business entities including **Customer, Order, Product, Shipment, Market, Department, Category, and Delivery**.
- These entities support analysis across **sales, profitability, customer behavior, product performance, and delivery operations**.
- Supplier and inventory entities are part of the target procurement model but are **not directly represented in the current DataCo fields**.
- The identified entities will guide the **data warehouse design and KPI development**.

# 11. Column Classification

In [27]:
classification = pd.DataFrame({

"Column":dataco.columns,

"Category":""

})

classification.head()

,Column,Category
0,Type,
1,Days for shipping (real),
2,Days for shipment (scheduled),
3,Benefit per order,
4,Sales per customer,


### Observation

- A column classification framework has been created to organize fields by **business role and analytical purpose**.
- This classification will help distinguish **dimensions, measures, identifiers, dates, and operational fields**.
- The classification will support **feature selection, KPI development, and data warehouse design** in subsequent stages.

# 12. External Data Mapping

In [28]:
mapping = pd.DataFrame({

"External Dataset":[

"Fuel Prices",

"Commodity Prices",

"Exchange Rate",

"Holiday",

"Weather"

],

"Join Key":[

"Date",

"Date",

"Date",

"Date + Country",

"Date + City"

],

"Business Purpose":[

"Transportation Cost",

"Raw Material Cost",

"International Procurement",

"Lead Time Analysis",

"Shipment Delay Analysis"

]

})

mapping

,External Dataset,Join Key,Business Purpose
0,Fuel Prices,Date,Transportation Cost
1,Commodity Prices,Date,Raw Material Cost
2,Exchange Rate,Date,International Procurement
3,Holiday,Date + Country,Lead Time Analysis
4,Weather,Date + City,Shipment Delay Analysis


### Observations

- External datasets provide important **contextual factors** for procurement and supply-chain analysis.
- **Fuel and commodity prices** can help explain transportation and raw-material cost changes.
- **Exchange rates** support analysis of international procurement exposure.
- **Holiday and weather data** can help identify factors affecting lead times and shipment delays.
- These datasets will be integrated using **date, country, and city-based keys** during the ETL stage.

# 13. Business Questions

The following business questions will be answered throughout the project.

1. Which suppliers are the most reliable?
2. Which suppliers are becoming risky?
3. Which products have unstable procurement costs?
4. Which commodity prices affect procurement the most?
5. Which suppliers should receive future purchase orders?
6. How do exchange rates affect procurement costs?
7. Does weather influence delivery delays?
8. What procurement cost can be expected next month?
9. Which inventory items are likely to stock out?
10. Which vendors provide the highest long-term value?

# 14. Next Steps

The next notebook will focus on:

- Data Cleaning
- Missing Value Treatment
- Duplicate Removal
- Data Type Conversion
- Outlier Detection
- Data Standardization

This prepares the datasets for feature engineering and predictive modeling.